# Heady™ Liquid Colab Node v5.9.0

**GPU-accelerated embedding, inference, and similarity compute node.**

This notebook runs as a Heady liquid node exposing:
- `/_heady/health` — Health check + GPU status
- `/_heady/embed` — Sentence transformer embeddings
- `/_heady/similarity` — Cosine similarity scoring
- `/_heady/phi-transform` — φ-scaled vector transformation

φ = 1.618033988749895 | ψ = 0.618033988749895

In [ ]:
# ── Cell 1: Install Dependencies ──
!pip install -q torch transformers sentence-transformers \
  pgvector psycopg2-binary fastapi uvicorn pyngrok numpy

In [ ]:
# ── Cell 2: Load Embedding Model ──
import torch
from sentence_transformers import SentenceTransformer
import numpy as np

PHI = 1.6180339887498949
PSI = 0.6180339887498949

# Detect GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'[heady] Device: {device} — {gpu_name}')

# Load sentence transformer
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f'[heady] Model loaded: all-MiniLM-L6-v2 on {device}')
print(f'[heady] Embedding dimensions: {model.get_sentence_embedding_dimension()}')

In [ ]:
# ── Cell 3: FastAPI Liquid Node Server ──
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
import time

app = FastAPI(title='Heady™ Liquid Colab Node', version='5.9.0')

class EmbedRequest(BaseModel):
    texts: List[str]
    normalize: bool = True

class SimilarityRequest(BaseModel):
    query: str
    candidates: List[str]
    threshold: float = 0.5

class PhiTransformRequest(BaseModel):
    vectors: List[List[float]]
    scale: float = PHI

# Track metrics
metrics = {'requests': 0, 'embeddings_generated': 0, 'start_time': time.time()}

@app.get('/_heady/health')
async def health():
    return {
        'status': 'ok',
        'node': 'colab',
        'gpu': gpu_name,
        'device': device,
        'model': 'all-MiniLM-L6-v2',
        'dims': model.get_sentence_embedding_dimension(),
        'phi': PHI,
        'uptime_seconds': round(time.time() - metrics['start_time'], 1),
        'metrics': metrics,
        'cuda_available': torch.cuda.is_available(),
        'cuda_memory': {
            'allocated_mb': round(torch.cuda.memory_allocated() / 1e6, 1) if torch.cuda.is_available() else 0,
            'reserved_mb': round(torch.cuda.memory_reserved() / 1e6, 1) if torch.cuda.is_available() else 0,
        } if torch.cuda.is_available() else None,
    }

@app.post('/_heady/embed')
async def embed(req: EmbedRequest):
    if not req.texts:
        raise HTTPException(status_code=400, detail='texts required')
    metrics['requests'] += 1
    start = time.time()
    vectors = model.encode(req.texts, normalize_embeddings=req.normalize, device=device)
    elapsed_ms = round((time.time() - start) * 1000, 1)
    metrics['embeddings_generated'] += len(req.texts)
    return {
        'vectors': vectors.tolist(),
        'dims': vectors.shape[1],
        'count': len(req.texts),
        'elapsed_ms': elapsed_ms,
        'device': device,
    }

@app.post('/_heady/similarity')
async def similarity(req: SimilarityRequest):
    metrics['requests'] += 1
    q_vec = model.encode([req.query], normalize_embeddings=True, device=device)
    c_vecs = model.encode(req.candidates, normalize_embeddings=True, device=device)
    scores = np.dot(c_vecs, q_vec.T).flatten().tolist()
    results = [
        {'text': c, 'score': round(s, 4), 'above_threshold': s >= req.threshold}
        for c, s in sorted(zip(req.candidates, scores), key=lambda x: -x[1])
    ]
    return {
        'query': req.query,
        'results': results,
        'threshold': req.threshold,
        'above_threshold_count': sum(1 for r in results if r['above_threshold']),
    }

@app.post('/_heady/phi-transform')
async def phi_transform(req: PhiTransformRequest):
    """Apply φ-scaled transformation to vectors (sacred geometry projection)"""
    metrics['requests'] += 1
    transformed = []
    for vec in req.vectors:
        arr = np.array(vec)
        phi_scaled = arr * req.scale
        magnitude = float(np.linalg.norm(phi_scaled))
        normalized = (phi_scaled / magnitude).tolist() if magnitude > 0 else phi_scaled.tolist()
        transformed.append({
            'original': vec,
            'phi_scaled': phi_scaled.tolist(),
            'normalized': normalized,
            'magnitude': round(magnitude, 6),
        })
    return {'transforms': transformed, 'scale': req.scale, 'phi': PHI}

print('[heady] FastAPI server defined — 4 endpoints ready')

In [ ]:
# ── Cell 4: Start Server with ngrok Tunnel ──
import nest_asyncio
nest_asyncio.apply()

# Configure ngrok (set your token in Colab secrets or environment)
import os
NGROK_TOKEN = os.environ.get('NGROK_AUTHTOKEN', '')

if NGROK_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8080)
    print(f'[heady] 🌐 Public URL: {public_url}')
    print(f'[heady] Health: {public_url}/_heady/health')
else:
    print('[heady] ⚠️ No NGROK_AUTHTOKEN — running locally on :8080')
    print('[heady] Set NGROK_AUTHTOKEN in Colab secrets for external access')

import uvicorn
print('[heady] 🚀 Starting Heady Liquid Colab Node on :8080')
uvicorn.run(app, host='0.0.0.0', port=8080)